# Match Hearings to CAP Records

| Item | Description |
|---|---|
| Goal | Link each sample hearing to its row in the Comparative Agendas Project (CAP) hearings dataset, so that T2.3.2 can compare the codebook with CAP's human topic codes. The link uses only metadata: Congress, committee, date, and title. The CAP topic columns are never read here, so the codebook, which was built without CAP, meets the human codes for the first time in T2.3.2. |
| Input | `US-Legislative_congressional_hearings-21.7.csv` in the T2.3 Raw folder, downloaded from the Comparative Agendas Project (comparativeagendas.net, US Congressional Hearings dataset, version 21.7); `hearing_summaries.parquet` from T1.2 for the sample's metadata. |
| Output | `cap_match.parquet` in the T2.3 Processed folder: one row per matched pair with its tier, title similarity, and date distance. |
| Sources | Jones, Bryan D., Frank R. Baumgartner, Sean M. Theriault, Derek A. Epp, Shruti Khandekar, Daniel Little. 2025. *Policy Agendas Project: Codebook*. Comparative Agendas Project, comparativeagendas.net. The hearings file is the US Congressional Hearings dataset, version 21.7, of the same project; CAP's topic codes are not read in this notebook. |
| Variables | • `tier`: 1 for a same-day pair with title agreement, 2 for a pair within three days or an undated pair with strong agreement, 3 for a best-in-block pair on title alone<br>• `sim`: the larger of Jaccard and containment between the two title word sets<br>• `date_diff`: days between the two dates, missing when CAP has no date |
| Run | First run 2026-08-22, which wrote the stored table. Rebuilt into a scratch copy on 2026-10-31 and found identical; the outputs shown are from that rebuild. CAP ends with the 117th Congress, so hearings from 2022 on cannot match by construction. |


## Step 1. Import packages


In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T2 - Topic Extraction and Stance Analysis/T2.3 - Validation of Topics")
NOTEBOOK_NAME = "T2.3.1 - Validation - Match Hearings to CAP Records"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
INPUT_FOLDER = SCRATCH_FOLDER / "T1 - Text Summarization/T1.2 - Process Entire Sample Congressional Hearings" / "Processed"
CAP_CSV = RAW_FOLDER / "comparative_agenda_project" / "US-Legislative_congressional_hearings-21.7.csv"
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Load the two sides

CAP side: rows whose primary or secondary committee is House Science (code 120) or Senate Commerce (code 207), from 1996 on. Only the matching columns are read; the topic columns stay unread. The file is not valid UTF-8, so it is read as Latin-1. Our side: the 2,352 sample hearings with title, chamber, Congress, and first held date.


In [4]:
SAFE_COLS = ["id", "source", "Chamber", "Congress", "year", "Month", "Days", "Committee1", "Committee2", "title", "StartDate", "CISYear"]

cap = pd.read_csv(CAP_CSV, encoding="latin-1", usecols=SAFE_COLS, low_memory=False)
for c in ("Committee1", "Committee2"):
    cap[c] = cap[c].astype(str).str.replace(r"\.0$", "", regex=True)
cap = cap.loc[(cap["year"] >= 1996) & (cap["Committee1"].isin(["120", "207"]) | cap["Committee2"].isin(["120", "207"]))].copy()
cap["committee"] = np.where(cap["Committee1"].isin(["120", "207"]), cap["Committee1"], cap["Committee2"])
cap["date"] = pd.to_datetime(cap["StartDate"], format="%m/%d/%Y", errors="coerce")

ours = pd.read_parquet(INPUT_FOLDER / "hearing_summaries.parquet", columns=["packageId", "title", "congress", "chamber", "year", "heldDate_first"])
ours["congress"] = ours["congress"].astype(int)
ours["date"] = pd.to_datetime(ours["heldDate_first"])
ours["side"] = ours["chamber"].map({"HOUSE": "120", "SENATE": "207"})  # the four joint hearings may match either side

print(f"CAP candidates : {len(cap):,} (dates present {cap['date'].notna().mean():.0%})")
print(f"SST hearings   : {len(ours):,}; matchable window up to the 117th Congress: {(ours['congress'] <= 117).sum():,}")


CAP candidates : 2,426 (dates present 76%)
SST hearings   : 2,352; matchable window up to the 117th Congress: 2,209


## Step 4. Score title similarity

Both titles become word sets: lower case, punctuation removed, common words dropped. Similarity is the larger of Jaccard and containment; containment covers the common case where one source shortens the other's long title.


In [5]:
STOP = {"the", "of", "and", "a", "an", "on", "in", "for", "to", "act", "hearing", "hearings", "oversight", "before",
        "committee", "s", "hr", "review", "examining", "fiscal", "year"}


def tokens(title):
    if not isinstance(title, str):
        return frozenset()
    words = re.sub(r"[^a-z0-9 ]", " ", title.lower()).split()
    return frozenset(w for w in words if w not in STOP and len(w) > 1)


def sim(a, b):
    if not a or not b:
        return 0.0
    inter = len(a & b)
    return max(inter / len(a | b), inter / min(len(a), len(b)))


cap["tok"] = cap["title"].map(tokens)
ours["tok"] = ours["title"].map(tokens)
print("empty word sets: CAP", (cap["tok"].map(len) == 0).sum(), "| ours", (ours["tok"].map(len) == 0).sum())


empty word sets: CAP 63 | ours 1


## Step 5. Accept pairs one to one, in tiers

Within each Congress and committee side, every pair is scored and accepted best first, each side used at most once. A same-day pair needs less title agreement than an undated one. The tiers are:

- Tier 1: same date and similarity at least 0.5, or similarity at least 0.85 within one day.
- Tier 2: within three days and similarity at least 0.35, or no CAP date and similarity at least 0.75.
- Tier 3: the remaining pairs with similarity at least 0.55.

T2.3.2 uses tiers 1 and 2.


In [6]:
pairs = []
for (cong, side), block in ours.groupby(["congress", "side"], dropna=False):
    cands = cap.loc[(cap["Congress"] == cong)]
    if side is not np.nan and isinstance(side, str):
        cands = cands.loc[cands["committee"] == side]
    for o in block.itertuples():
        for c in cands.itertuples():
            s = sim(o.tok, c.tok)
            if s == 0.0:
                continue
            dd = abs((o.date - c.date).days) if pd.notna(c.date) else None
            if (dd == 0 and s >= 0.5) or (dd is not None and dd <= 1 and s >= 0.85):
                tier = 1
            elif (dd is not None and dd <= 3 and s >= 0.35) or (dd is None and s >= 0.75):
                tier = 2
            elif s >= 0.55:
                tier = 3
            else:
                continue
            pairs.append({"packageId": o.packageId, "cap_id": c.id, "cap_source": c.source, "tier": tier, "sim": s, "date_diff": dd})

pairs = pd.DataFrame(pairs).sort_values(["tier", "sim"], ascending=[True, False])
used_o, used_c, keep = set(), set(), []
for r in pairs.itertuples():
    if r.packageId in used_o or r.cap_id in used_c:
        continue
    used_o.add(r.packageId)
    used_c.add(r.cap_id)
    keep.append(r.Index)
match = pairs.loc[keep].reset_index(drop=True)
match.to_parquet(PROCESSED_FOLDER / "cap_match.parquet", index=False)

in_window = ours.loc[ours["congress"] <= 117, "packageId"]
print(f"matched: {len(match):,} of {len(in_window):,} in-window hearings ({len(match) / len(in_window):.0%})")
print(match["tier"].value_counts().sort_index().to_string())


matched: 1,796 of 2,209 in-window hearings (81%)
tier
1    1699
2      64
3      33


## Step 6. Check the matches

One-to-one by construction, match rates by year, and the weakest accepted pairs printed for reading.


In [7]:
back = pd.read_parquet(PROCESSED_FOLDER / "cap_match.parquet")
assert not back["packageId"].duplicated().any()
assert not back["cap_id"].duplicated().any()

m = ours.merge(back, on="packageId", how="left")
rate = m.loc[m["congress"] <= 117].groupby("year")["cap_id"].agg(lambda s: s.notna().mean())
print("1) match rate by year (in the CAP window):")
print("   " + " ".join(f"{y}:{r:.0%}" for y, r in rate.items()))
print(f"2) overall in-window rate: {m.loc[m['congress'] <= 117, 'cap_id'].notna().mean():.0%} | tiers: {back['tier'].value_counts().sort_index().to_dict()}")
print(f"3) out of window (118th Congress on): {(m['congress'] > 117).sum():,} hearings, unmatchable by construction")

weak = (back.sort_values("sim").head(8).merge(ours[["packageId", "title", "date"]], on="packageId")
        .merge(cap[["id", "title", "date"]].rename(columns={"id": "cap_id", "title": "cap_title", "date": "cap_date"}), on="cap_id"))
print("\n4) weakest accepted pairs:")
for r in weak.itertuples():
    print(f"   T{r.tier} sim={r.sim:.2f} dd={r.date_diff}")
    print(f"     SST: {str(r.title)[:90]}")
    print(f"     CAP: {str(r.cap_title)[:90]}")


1) match rate by year (in the CAP window):
   1996:100% 1997:100% 1999:98% 2000:100% 2001:97% 2002:100% 2003:98% 2004:90% 2005:100% 2006:100% 2007:99% 2008:99% 2009:87% 2010:98% 2011:89% 2012:100% 2013:100% 2014:98% 2015:97% 2016:87% 2017:73% 2018:25% 2019:22% 2020:37% 2021:18% 2022:2%
2) overall in-window rate: 81% | tiers: {1: 1699, 2: 64, 3: 33}
3) out of window (118th Congress on): 143 hearings, unmatchable by construction

4) weakest accepted pairs:
   T3 sim=0.57 dd=nan
     SST: FOSSIL ENERGY RESEARCH: ENABLING OUR CLEAN ENERGY FUTURE
     CAP: From Lab to Market: Accelerating Our Progress Toward Economic Recovery and a Clean Energy 
   T3 sim=0.60 dd=nan
     SST: SCIENCE AND ENERGY RESEARCH INFRASTRUCTURE NEEDS OF THE U.S. DEPARTMENT OF ENERGY
     CAP: Department of Energy Science for the Future ActDepartment of Energy Science for the Future
   T1 sim=0.67 dd=0.0
     SST: NASA FY 2008 BUDGET
     CAP: NASA FY2008 Budget
   T3 sim=0.67 dd=5179.0
     SST: PIPELINE SAFETY IN T